# Data Science in Psychology and Neuroscience

## Class info:
* Week #10
* Day: March 24, 2026
* Time: 9:30—10:45 AM
* Location: Logan Hall 125
* <a href="https://forms.microsoft.com/r/26vAcJWrwH">Click here to submit your attendance for Week 10, Question 17!</a>
  
## Instructor info:
* Dr. Jeremy Hogeveen
* jhogeveen@unm.edu
* Logan Hall 281 (Office Hours By Appointment)
  
## Syllabus:
* <a href="https://www.dropbox.com/scl/fi/6fs6fi4kvkwtxn7j7x8ua/PSY450650_DSPN_Spring2026_Syllabus.pdf?rlkey=148e5t4ah8q2n1daclt7mp0h6&dl=0">Download here</a>

## Today's topic:
* So, you want to model some data...
    * Examples.

# Section 1: So you want to model some data?

<img src="img/what-does-that-mean-david.gif" width=300>

## 1.1 What did you find?
* Let's say you have...
1. Acquired some data
2. Wrangled the data
3. Computed descriptive statistics (central tendencies, variance, etc.)
4. Generated some visualizations
* Remaining questions: What did you find?

### __Inferential (or predictive) modeling is used to generate an interim answer about your pre-defined research questions / hypotheses__.

## Which model is appropriate?

<img src="img/decision_tree.png" width=450>

__Key questions:__
1. Are the main outcome measures you're interested in continuous or categorical?
2. If continuous, is your research question about associations or differences?
3. If continuous + associations...
    - Independent and dependent variable?
4. If continuous + differences...
    - Differences between what?

# Section 2: Modeling Exercise 1

### 2.1 Background: _Gebotys and Roberts (1989) were interested in examining the effects of several variables on the “seriousness rating of the crime”. The variables to be examined within this example are “age” (in years), the “amount of television news watched in hours per week” (i.e., ‘tvnews’), and whether or not the respondents had experience being a victim of crime in the past (i.e., 'experience')._

In [4]:
suppressPackageStartupMessages(library(tidyverse))

In [5]:
df <- tibble(pid  = c(1,2,3,4,5,6,7,8,9,10,11),
             age = c(10,25,26,25,30,34,40,40,40,25,80),
             tv_news = c(4.0,5.0,5.0,4.5,6.0,7.0,5.5,6.0,7.0,8.5,9.0),
             experience = as_factor(c(0,0,0,0,0,1,0,1,1,1,1)),
             crime_seriousness = c(21,28,27,26,33,36,31,35,41,80,95))
# df

### 2.2 Exercise 1, research question 1: Is there an association between TV news viewership and crime seriousness ratings?

### __Linear relationship: Determine whether the variables change together at a constant rate__

### __Monotonic relationship: the variables change together, but not _necessarily_ at a constant rate.__

### 2.2 Exercise 1, research question 2: Does age predict tv news viewership?

### __Linear regression: Does $y$ change at a constant rate as a function of $x$.__

### 2.3 Exercise 1, research question 3: Does the effect of TV watching on crime seriousness ratings vary as a function of age?

In [28]:
# Creating an interaction plotting function, inspired by sjPlot

# For categorical moderator term
plot_interaction <- function(data, outcome, predictor, moderator) {
  
  # Force the moderator to be a factor (categorical)
  data[[moderator]] <- as.factor(data[[moderator]])
  
  # Fit the model
  formula_str <- paste(outcome, "~", predictor, "*", moderator)
  model <- lm(as.formula(formula_str), data = data)
  
  # Find the min and max of the predictor, and get all categories of the moderator
  grid_data <- expand.grid(
    pred_vals = seq(min(data[[predictor]], na.rm = TRUE), 
                    max(data[[predictor]], na.rm = TRUE), 
                    length.out = 100),
    mod_vals = levels(data[[moderator]])
  )
  
  # Rename grid columns so the predict() function recognizes them
  names(grid_data) <- c(predictor, moderator)
  
  # Pull model predictions and confidence intervals
  predictions <- predict(model, newdata = grid_data, interval = "confidence")
  plot_data <- cbind(grid_data, predictions)
  
  # Plot the interaction!
  # Passing string variables into R functions is difficult, we use the .data[[variablestring]] approach based on advice from stack overflow
  ggplot(plot_data, aes(x = .data[[predictor]], y = fit, 
                        color = .data[[moderator]], 
                        fill = .data[[moderator]])) +
    geom_ribbon(aes(ymin = lwr, ymax = upr), alpha = 0.2, color = NA) +
    geom_line(linewidth = 1.2) +
    geom_point(data = data, aes(x = .data[[predictor]], y = .data[[outcome]]), alpha = 0.4) +
    labs(
      title = paste(predictor, "and", moderator," interaction plot."),
      y = paste("Predicted", outcome),
      color = moderator,
      fill = moderator
    ) +
    theme_classic()
}

plot_continuous_interaction <- function(data, outcome, predictor, moderator) {
  formula_str <- paste(outcome, "~", predictor, "*", moderator)
  model <- lm(as.formula(formula_str), data = data)
  # Using "spotlight" approach to identify low, med, and high mod values
  mod_mean <- mean(data[[moderator]], na.rm = TRUE)
  mod_sd   <- sd(data[[moderator]], na.rm = TRUE)
  spotlight_vals <- c(mod_mean - mod_sd, mod_mean, mod_mean + mod_sd)
  spotlight_labels <- c("Low (-1 SD)", "Mean", "High (+1 SD)") 
  # Create grid using only the spotlight values of moderator
  grid_data <- expand.grid(
    pred_vals = seq(min(data[[predictor]], na.rm = TRUE), 
                    max(data[[predictor]], na.rm = TRUE), 
                    length.out = 100),
    mod_vals = spotlight_vals
  )
  names(grid_data) <- c(predictor, moderator)
  # Pull predictions and wrangle data together 
  predictions <- predict(model, newdata = grid_data, interval = "confidence")
  plot_data <- cbind(grid_data, predictions)
  
  # Convert the moderator in the PLOT DATA to a factor for plotting
  plot_data[[moderator]] <- factor(plot_data[[moderator]], 
                                   levels = spotlight_vals, 
                                   labels = spotlight_labels)
  # Plot the interaction
  ggplot(plot_data, aes(x = .data[[predictor]], y = fit, 
                        color = .data[[moderator]], 
                        fill = .data[[moderator]])) +
    geom_ribbon(aes(ymin = lwr, ymax = upr), alpha = 0.2, color = NA) +
    geom_line(linewidth = 1.2) +
    labs(
      title = paste(predictor, "and", moderator," interaction plot."),
      y = paste("Predicted", outcome),
      color = moderator,
      fill = moderator
    ) +
    theme_classic()
}